In [1]:
# pip install pandas
# pip install matplotlib

In [2]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from typing import List, Dict, Set, Optional
from dataclasses import dataclass
import random

# Use WenQuanYi Zen Hei as the default Chinese font
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei']
plt.rcParams['axes.unicode_minus'] = False  # Avoid issues displaying the minus sign


In [3]:
@dataclass
class ClarificationNode:
    """Data structure for a clarification node"""
    id: str
    text: str
    aspect: str
    depth: int
    parent_id: Optional[str] = None
    answers: Optional[List[str]] = None


In [4]:
# 2. Knowledge base data
def create_knowledge_base() -> Dict:
    """Create the knowledge base"""
    return {
        "abilities": {
            "combat": {
                "physical": ["Vajra Body", "Immense Strength", "Iron Sinews"],
                "magical": ["72 Transformations", "Somersault Cloud", "Body-Binding Spell"],
                "weapon": ["Staff Mastery", "Weapon Master", "Magic Weapon Control"]
            },
            "perception": {
                "detection": ["Golden Eyes", "Demon Aura Detection", "Qi Sensing"],
                "insight": ["Insight into Nature", "Demon Discernment", "Spirit Sense"]
            }
        },
        "relationships": {
            "allies": {
                "disciples": ["Red Boy", "Mingyue", "Six-Eared Macaque"],
                "friends": ["Erlang Shen", "Nezha", "Third Prince of the Dragon King"]
            },
            "enemies": {
                "heavenly": ["Heavenly Soldiers and Generals", "Erlang Shen's Forces"]
            }
        }
    }


In [5]:
# 3. Template management
def create_question_templates(character_name: str) -> Dict[str, Dict[str, str]]:
    """Create the question templates"""
    return {
        "abilities": {
            "root": f"Which kind of ability of {character_name} would you like to learn about?",
            "combat": f"Regarding combat abilities, which aspect of {character_name} are you more interested in?",
            "perception": f"Regarding {character_name}'s perception abilities, what specifically would you like to know?"
        },
        "relationships": {
            "root": f"Whose relationship with {character_name} would you like to learn about?",
            "allies": f"Among {character_name}'s allies, which kind of relationship would you like to know about?",
            "enemies": f"Regarding {character_name}'s opponents, which faction are you interested in?"
        }
    }


In [6]:
# 4. Question analysis
def identify_main_aspects(question: str) -> Set[str]:
    """Identify the main aspects of the question"""
    aspects = set()
    keywords = {
        "abilities": ["ability", "abilities", "skill", "skills", "spell", "power"],
        "relationships": ["relationship", "friend", "enemy", "master-disciple", "befriend"]
    }

    for aspect, words in keywords.items():
        if any(word in question for word in words):
            aspects.add(aspect)

    if not aspects:
        aspects.add("abilities")

    return aspects

def identify_character(question: str, characters: List[str]) -> Optional[str]:
    """Identify the character mentioned in the question"""
    for character in characters:
        if character in question:
            return character
    return None


In [7]:
# 5. Tree construction
def build_clarification_tree(
    question: str,
    character: str,
    knowledge_base: Dict,
    templates: Dict[str, Dict[str, str]]
) -> nx.DiGraph:
    """Build the question clarification tree"""
    G = nx.DiGraph()

    # Add the root node
    root_id = "root"
    G.add_node(root_id, text=question, depth=0)

    # Analyze the question's topics
    main_aspects = identify_main_aspects(question)

    # Build each layer of the tree
    for aspect in main_aspects:
        if aspect not in knowledge_base:
            continue

        # Layer 1: main aspects
        node_id = f"{aspect}_root"
        G.add_node(node_id,
                  text=templates[aspect]["root"],
                  depth=1)
        G.add_edge(root_id, node_id)

        # Layer 2: sub-aspects
        for sub_aspect, sub_data in knowledge_base[aspect].items():
            sub_node_id = f"{aspect}_{sub_aspect}"
            template_text = templates[aspect].get(
                sub_aspect,
                f"What would you like to know about {sub_aspect}?"
            )
            G.add_node(sub_node_id, text=template_text, depth=2)
            G.add_edge(node_id, sub_node_id)

            # Layer 3: specific details
            for detail_type, details in sub_data.items():
                detail_node_id = f"{aspect}_{sub_aspect}_{detail_type}"
                detail_text = f"Would you like to know about {detail_type}: {', '.join(details)}?"
                G.add_node(detail_node_id,
                         text=detail_text,
                         depth=3,
                         answers=details)
                G.add_edge(sub_node_id, detail_node_id)

    return G


In [8]:
# 6. Visualization
def visualize_tree(G: nx.DiGraph, figsize=(15, 10)):
    """Visualize the question clarification tree"""
    plt.figure(figsize=figsize)

    pos = nx.spring_layout(G, k=2, iterations=50)
    depths = nx.get_node_attributes(G, 'depth')
    colors = ['lightblue', 'lightgreen', 'lightyellow', 'lightpink']

    for depth in range(max(depths.values()) + 1):
        nodes_at_depth = [node for node, d in depths.items() if d == depth]
        nx.draw_networkx_nodes(G, pos,
                             nodelist=nodes_at_depth,
                             node_color=colors[depth],
                             node_size=2000)

    nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True)
    labels = nx.get_node_attributes(G, 'text')
    nx.draw_networkx_labels(G, pos, labels, font_size=8, font_weight='bold')

    plt.title("Question Clarification Tree")
    plt.axis('off')
    plt.tight_layout()
    plt.show()


In [9]:
# 7. Usage example
def process_question(question: str):
    """The full pipeline for processing a question"""
    # Initialize the knowledge base and character list
    kb = create_knowledge_base()
    characters = ["Sun Wukong", "Erlang Shen", "Zhu Bajie"]

    # Identify the character
    character = identify_character(question, characters)
    if not character:
        return "Could not identify the character in the question"

    # Create the templates
    templates = create_question_templates(character)

    # Build and visualize the tree
    tree = build_clarification_tree(question, character, kb, templates)
    visualize_tree(tree)

    return tree


In [ ]:
# Simple usage
question = "What abilities does Sun Wukong have?"
tree = process_question(question)


In [ ]:
# Or use it step by step
kb = create_knowledge_base()
templates = create_question_templates("Sun Wukong")
tree = build_clarification_tree(question, "Sun Wukong", kb, templates)
visualize_tree(tree)
